# Nazih · Training Set Size — GitHub → Google Colab

**Repository-first workflow:** no local study ZIP upload. The code and archived textual results are fetched directly from the isolated GitHub branch.

**Important:** The official KSDD2 image archive is distributed separately under CC BY-NC-SA 4.0. The historical 70 source images are recovered from that official archive and checked against their recorded SHA256 hashes. New training will not run unless the original archive hash and CPU software versions match the archived reference. **Do not select T4 GPU for a CPU-reference comparison.** No new training has yet been completed here.


In [ ]:
import os, subprocess, sys, pathlib, hashlib
REPO='https://github.com/ID9999999999999/DL-Computer-vision.git'
BRANCH='nazih-training-set-size-20261109'
CLONE=pathlib.Path('/content/nazih-training-set-size')
BASE=CLONE/'COURSEWORK_NAZIH'
if CLONE.exists() and (CLONE/'.git').exists():
    print('Repository already cloned; preserving existing runtime files:',BASE)
else:
    assert not CLONE.exists(), 'Unexpected existing directory. Inspect it first.'
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(CLONE)],check=True)
print('Repository ready:',BASE)


## 1. Obtain the official KSDD2 dataset, validate its SHA256, recover the original 70 training images

The dataset may be downloaded from its original source below; availability is controlled by the publisher. If automatic download fails, download it from [the official page](https://www.vicos.si/resources/kolektorsdd2/) and place the *unmodified* `KolektorSDD2.zip` in `/content` using the Colab Files sidebar. No other source will be silently substituted.


In [ ]:
import urllib.request, zipfile, json, hashlib
archive=pathlib.Path('/content/KolektorSDD2.zip')
expected='edcdb486809b24f1d17b785e30c52fafc5999554dd5fe18ddf77b61ceb6f36a8'
if not archive.exists():
    url='https://data.vicos.si/datasets/KSDD/KolektorSDD2.zip'
    print('Attempting official archive download:',url)
    try:
        urllib.request.urlretrieve(url,str(archive)+'.part')
        pathlib.Path(str(archive)+'.part').replace(archive)
    except Exception as e:
        pathlib.Path(str(archive)+'.part').unlink(missing_ok=True)
        raise RuntimeError('Official download failed. Manually obtain the ORIGINAL ZIP from the ViCoS dataset page and upload it to /content/KolektorSDD2.zip.') from e
h=hashlib.sha256()
with archive.open('rb') as f:
    for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
assert h.hexdigest()==expected, ('Original archive SHA256 mismatch',h.hexdigest())
manifest=json.loads((BASE/'input/target_manifest.json').read_text())
train_dir=BASE/'input/train';train_dir.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for r in manifest:
        data=z.read(r['zip_path'])
        assert hashlib.sha256(data).hexdigest()==r['sha256'],r['zip_path']
        p=train_dir/pathlib.Path(r['zip_path']).name
        if p.exists():
            assert hashlib.sha256(p.read_bytes()).hexdigest()==r['sha256']
        else:p.write_bytes(data)
print('Verified official ZIP and original 70 images:',len(manifest))


## 2. Run integrity and methodological checks (does not train)

In [ ]:
def run_script(name,*args):
    subprocess.run([sys.executable,str(BASE/'scripts'/name),*map(str,args)],cwd=BASE,check=True)
run_script('preflight.py')
run_script('test_pipeline.py')
run_script('prepare.py','--archive',archive)
print('Preflight, unit tests and nested 140/280/420 manifests passed.')


## 3. Verify CPU environment against historical 70-image reference

**Mandatory guard:** old reference: Python 3.12.14, PyTorch 2.15.0+cpu, torchvision 0.29.2+cpu, NumPy 2.5.4, Pillow 13.0.0. Colab's default environment may differ; in that case stop rather than claiming a matched comparison. Do not use GPU for these reference comparisons. Do not blindly run `pip install` in a mismatched runtime.


In [ ]:
import platform
print('Python:',platform.python_version())
sys.path.insert(0,str(BASE/'scripts'))
from train import test_environment
assert platform.python_version()=='3.12.14', 'Python version differs from historical reference; do not run new matched experiments.'
assert not os.environ.get('CUDA_VISIBLE_DEVICES') or os.environ['CUDA_VISIBLE_DEVICES'] in ('','-1'), 'Disable GPU for this CPU reference study.'
print('Matched environment:',test_environment(strict=True))


## 4. Train the next NEW model (size 140, first seed, CE only)

The older 70-image models are never retrained. Run only when all previous cells passed. The remaining 17 combinations can be resumed with `scripts/runner.py`. Results in `/content` disappear when the Colab runtime resets: copy `results/` to Google Drive after each successful run.


In [ ]:
run_script('train.py','--archive',archive,'--size',140,'--seed',20261028,'--loss','ce')
print('First new training completed. Save your results directory before leaving Colab.')


## 5. Save results to Drive; optional full resume

The full runner command below should only be launched after manually inspecting the first training and ensuring persistent storage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
backup=pathlib.Path('/content/drive/MyDrive/Nazih_Training_Set_Size_Results')
backup.mkdir(parents=True,exist_ok=True)
for f in (BASE/'results').glob('*'):
    if f.is_file():shutil.copy2(f,backup/f.name)
print('Backed up runtime result files:',backup)
